# 8 · Rating models, and the "rising above yourself" test

Turns ball values into player ratings, with several methods (a ridge plus-minus model, empirical Bayes variants, a
combined profile). It also runs the key test for the brief's "compared with their usual level": is performing better
under pressure than your own normal a repeatable skill? (In this data, it is not.)

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

<details><summary>Technical notes</summary>

P6 - Player rating models (R-1 ridge APM, R-2 empirical Bayes x3, R-3 composite) fitted on all data.  
Also: the clutch-spread test (is there real between-player variance in the pressure-vs-usual gap?),  
model agreement (Kendall tau), and a grouped-CV check of the ridge penalties.
</details>

In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from rrlib.common import CFG, load, save_table, save_json, load_json, get_logger, Timer
from rrlib import ratings as RL

log = get_logger("p06_ratings")
v = load("values")
players = load("players").set_index("player_id")
comp = load_json("p05_components")

summary = {}
all_tabs = {}
for side in ("bat", "bowl"):
    tabs = {}
    with Timer(log, f"{side} ratings"):
        for met in ("v1", "v2", "v3", "v4"):
            d = RL.side_frame(v, side, met)
            e, k, tau2 = RL.eb_shrink(d)
            dl, tau_d, mu_d = RL.delta_shrink(d)
            ep, kp, tau2p = RL.eb_shrink(d[d.pres])
            summary[f"{side}_{met}"] = {"k_all": k, "tau_all": np.sqrt(tau2), "k_pressure": kp, "tau_pressure": np.sqrt(tau2p),
                                        "tau_clutch": np.sqrt(tau_d), "league_gap": mu_d,
                                        "sigma_ball": float(d.val.std()),
                                        "clutch_share_of_pressure_var": tau_d / max(tau2p, 1e-12)}
        for model in ("R2-p_only", "R2-joint", "R2-weighted"):
            tabs[model] = RL.r2_ratings(v, side, "v1", model.split("-")[1])
        tabs["R1"] = RL.r1_ratings(v, side, "v1")
        tabs["R3"] = RL.r3_ratings(v, side, comp)
    # agreement among players with >= 60 pressure balls
    base = tabs["R2-joint"]
    pool = base.index[base.n_p >= 60]
    names = list(tabs)
    M = pd.DataFrame(index=names, columns=names, dtype=float)
    for a in names:
        for b_ in names:
            x, y = tabs[a].rating.reindex(pool), tabs[b_].rating.reindex(pool)
            okk = x.notna() & y.notna()
            M.loc[a, b_] = kendalltau(x[okk], y[okk])[0]
    save_table(M.reset_index().rename(columns={"index": "model"}), f"rating_agreement_{side}")
    log.info(f"{side} model agreement (Kendall tau, players with >=60 pressure balls, n={len(pool)}):\n{M.round(2).to_string()}")
    # combined table
    T = pd.DataFrame(index=base.index)
    T["name"] = players.name.reindex(T.index)
    T["n_pressure"] = base.n_p; T["n_other"] = base.n_u
    for nm, t in tabs.items():
        T[nm] = t.rating.reindex(T.index)
    T["R1_usual"] = tabs["R1"].usual.reindex(T.index); T["R1_clutch"] = tabs["R1"].clutch.reindex(T.index)
    T["gap_raw"] = base.gap_raw; T["gap_shrunk"] = base.gap_shrunk; T["gap_reliability"] = base.gap_rel
    T = T.sort_values("R2-joint", ascending=False)
    save_table(T.reset_index().rename(columns={"index": "player_id", "pid": "player_id"}), f"ratings_{side}_all_models")
    all_tabs[side] = T
    log.info(f"{side}: top 12 by R2-joint\n" + T[T.n_pressure >= 30].head(12)[["name", "n_pressure", "R2-joint", "R1", "R3", "R2-p_only", "gap_shrunk"]].round(3).to_string())

S = pd.DataFrame(summary).T
save_table(S.reset_index().rename(columns={"index": "side_metric"}), "clutch_spread")
log.info("Variance decomposition (per ball, runs or WP units):\n" + S.round(4).to_string())

15:45:20 [bat ratings] 1.8s


15:45:20 bat model agreement (Kendall tau, players with >=60 pressure balls, n=176):
             R2-p_only  R2-joint  R2-weighted    R1    R3
R2-p_only         1.00      0.40         0.68  0.40  0.47
R2-joint          0.40      1.00         0.57  0.93  0.31
R2-weighted       0.68      0.57         1.00  0.56  0.43
R1                0.40      0.93         0.56  1.00  0.31
R3                0.47      0.31         0.43  0.31  1.00


15:45:20 bat: top 12 by R2-joint
                       name  n_pressure  R2-joint     R1     R3  R2-p_only  gap_shrunk
pid                                                                                   
50284  Vaibhav Sooryavanshi       200.0     0.254  0.245  0.095      0.080        -0.0
6396              Tim David       280.0     0.230  0.221  0.080      0.079        -0.0
6281        Abhishek Sharma       503.0     0.198  0.189  0.067      0.033        -0.0
4260       Heinrich Klaasen       189.0     0.171  0.158  0.096      0.079         0.0
5039          Rajat Patidar       178.0     0.167  0.156  0.017      0.043        -0.0
1318          Andre Russell       104.0     0.153  0.144  0.010      0.003        -0.0
6726       Romario Shepherd       131.0     0.140  0.130 -0.025      0.007        -0.0
1699       Suryakumar Yadav       570.0     0.137  0.118  0.076      0.084         0.0
143          Dinesh Karthik       135.0     0.115  0.093  0.033      0.021        -0.0
3125      

15:45:22 [bowl ratings] 1.8s


15:45:22 bowl model agreement (Kendall tau, players with >=60 pressure balls, n=186):
             R2-p_only  R2-joint  R2-weighted    R1    R3
R2-p_only         1.00      0.33         0.72  0.32  0.48
R2-joint          0.33      1.00         0.47  0.90  0.28
R2-weighted       0.72      0.47         1.00  0.45  0.47
R1                0.32      0.90         0.45  1.00  0.27
R3                0.48      0.28         0.47  0.27  1.00


15:45:22 bowl: top 12 by R2-joint
                     name  n_pressure  R2-joint     R1     R3  R2-p_only  gap_shrunk
pid                                                                                 
2933       Jasprit Bumrah       383.0     0.164  0.188  0.036      0.051        -0.0
4641          Lungi Ngidi       141.0     0.117  0.137  0.017      0.008        -0.0
6256   Varun Chakravarthy       442.0     0.116  0.143  0.012      0.015        -0.0
7920         Nathan Ellis       227.0     0.110  0.131  0.023      0.046        -0.0
34485      Sufiyan Muqeem       127.0     0.103  0.111  0.028      0.081         0.0
4856       Benjamin Sears       117.0     0.102  0.107  0.003      0.089         0.0
781          Sunil Narine       353.0     0.099  0.128  0.035      0.055         0.0
3644        Kuldeep Yadav       319.0     0.091  0.116  0.014      0.009        -0.0
39657       Eshan Malinga       194.0     0.086  0.103  0.006      0.032         0.0
830        Josh Hazlewood      

15:45:22 Variance decomposition (per ball, runs or WP units):
             k_all  tau_all  k_pressure  tau_pressure  tau_clutch  league_gap  sigma_ball  clutch_share_of_pressure_var
bat_v1    666.4552   0.0938    869.2235        0.0731      0.0000      0.0066      2.4211                        0.0000
bat_v2    644.0489   0.0012    791.4541        0.0021      0.0014      0.0003      0.0309                        0.4544
bat_v3    441.7338   0.0386    333.1089        0.0459      0.0076     -0.0038      0.8105                        0.0271
bat_v4    603.1811   0.0797    572.2322        0.0738      0.0383      0.0001      1.9573                        0.2701
bowl_v1  1088.5481   0.0725    869.9514        0.0722      0.0000     -0.0120      2.3921                        0.0000
bowl_v2  1519.0604   0.0008    801.5552        0.0021      0.0016     -0.0003      0.0306                        0.5899
bowl_v3   450.9233   0.0251    416.8770        0.0268      0.0000      0.0037      0.5321         

#### Ridge penalty check: grouped 5-fold CV of held-out ball value, multipliers of the Bayesian penalty

In [3]:
folds = load("folds").set_index("match_id").fold
v["fold"] = v.match_id.map(folds)
rows = []
with Timer(log, "R1 penalty CV"):
    for km in (0.25, 0.5, 1.0, 2.0, 4.0):
        se, se0 = 0.0, 0.0
        for k in range(5):
            tr, te = v[v.fold != k], v[v.fold == k]
            fit = RL.r1_fit(tr, "v1", km)
            yb = -te.v1_bowl.values
            yb = yb - (-tr.v1_bowl.values).mean()
            pb = (fit["bat"].usual.reindex(te.striker_id).fillna(0).values * te.faced.values
                  + fit["bat"].clutch.reindex(te.striker_id).fillna(0).values * te.faced.values * te.is_pressure.values
                  - fit["bowl"].usual.reindex(te.bowler_id).fillna(0).values
                  - fit["bowl"].clutch.reindex(te.bowler_id).fillna(0).values * te.is_pressure.values)
            se += ((yb - pb) ** 2).sum(); se0 += (yb ** 2).sum()
        rows.append({"penalty_multiplier": km, "heldout_mse_reduction_pct": 100 * (1 - se / se0)})
        log.info(f"R1 penalty x{km}: held-out MSE reduction {100 * (1 - se / se0):.4f}%")
save_table(pd.DataFrame(rows), "r1_penalty_cv")
save_json(summary, "p06_summary")
log.info("P6 done")

15:45:25 R1 penalty x0.25: held-out MSE reduction 0.0082%


15:45:27 R1 penalty x0.5: held-out MSE reduction 0.0490%


15:45:29 R1 penalty x1.0: held-out MSE reduction 0.0637%


15:45:31 R1 penalty x2.0: held-out MSE reduction 0.0582%


15:45:33 R1 penalty x4.0: held-out MSE reduction 0.0428%


15:45:33 [R1 penalty CV] 10.7s


15:45:33 P6 done
